# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [00:41<00:00, 13.87s/it]


In [3]:
len(deals)

30

In [4]:
deals[10].describe()

'Title: Computer Deals at Costco: Up to $400 off + shipping varies\nDetails: Take up to hundreds off laptops, desktops, monitors, and more. We\'ve pictured the ASUS ROG Strix G18 Ultra 9 18" Gaming Laptop for $1,599.99 ($400 off). Shop Now at Costco\nFeatures: \nURL: https://www.dealnews.com/Computer-Deals-at-Costco-Up-to-400-off-shipping-varies/21812737.html?iref=rss-c39'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [5]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [6]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [7]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Celestron PowerSeeker 127EQ Telescope for $164 + free shipping
Details: The on-page coupon makes it the best price we found by $20. Buy Now at Amazon
Features: 127mm aperture  1000mm focal length  Model: 21049
URL: https://www.dealnews.com/products/Celestron/Celestron-Power-Seeker-127-EQ-Telescope/409562.html?iref=rss-c142

Title: Google Pixel 10 Smartphones at Visible: 50% off

In [8]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='Celestron PowerSeeker 127EQ is a Newtonian reflector telescope with a 127 mm (5-inch) aperture and a 1000 mm focal length, mounted on an equatorial mount suitable for beginner to intermediate amateur astronomy. The optical tube provides good light-gathering for planetary and lunar observation as well as brighter deep-sky objects; the equatorial mount allows for easier tracking of celestial objects when polar-aligned. It typically includes eyepieces and a finderscope for framing targets and comes in a portable package for backyard observing and introductory astrophotography.', price=164.0, url='https://www.dealnews.com/products/Celestron/Celestron-Power-Seeker-127-EQ-Telescope/409562.html?iref=rss-c142'), Deal(product_description="Samsung DU9000 Series UN98DU9000FXZA is a 98-inch 4K (3840x2160) Crystal UHD LED smart TV featuring HDR10+ support and a 120 Hz refresh rate for smoother motion. It runs Samsung's Tizen smart TV platform with buil

In [9]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


Celestron PowerSeeker 127EQ is a Newtonian reflector telescope with a 127 mm (5-inch) aperture and a 1000 mm focal length, mounted on an equatorial mount suitable for beginner to intermediate amateur astronomy. The optical tube provides good light-gathering for planetary and lunar observation as well as brighter deep-sky objects; the equatorial mount allows for easier tracking of celestial objects when polar-aligned. It typically includes eyepieces and a finderscope for framing targets and comes in a portable package for backyard observing and introductory astrophotography.
164.0
https://www.dealnews.com/products/Celestron/Celestron-Power-Seeker-127-EQ-Telescope/409562.html?iref=rss-c142

Samsung DU9000 Series UN98DU9000FXZA is a 98-inch 4K (3840x2160) Crystal UHD LED smart TV featuring HDR10+ support and a 120 Hz refresh rate for smoother motion. It runs Samsung's Tizen smart TV platform with built-in voice assistant compatibility (Alexa and Google Assistant), and includes multiple in

In [10]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [11]:
from agents.scanner_agent import ScannerAgent

In [12]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [13]:
result

DealSelection(deals=[Deal(product_description='Celestron PowerSeeker 127EQ is a refractor/reflector hybrid telescope with a 127mm aperture and 1000mm focal length, offering substantial light-gathering for deep-sky and planetary viewing. It includes an equatorial mount for tracking celestial objects, making it suitable for beginner-to-intermediate amateur astronomers who want manual tracking and higher magnification capability. The optical specifications support detailed views of the Moon, planets, and brighter deep-sky objects.', price=164.0, url='https://www.dealnews.com/products/Celestron/Celestron-Power-Seeker-127-EQ-Telescope/409562.html?iref=rss-c142'), Deal(product_description='Samsung DU9000 Series UN98DU9000FXZA is a 98-inch 4K UHD Crystal display with HDR10+ support and a 120Hz refresh rate for smoother motion. It runs Tizen OS smart platform with built-in voice assistants (Alexa and Google Assistant), and offers multiple connectivity options including three HDMI inputs and tw

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [ ]:
load_dotenv(override=True)

In [ ]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [ ]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
push("MASSIVE DEAL!!")

In [ ]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

In [ ]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")